In [ ]:
# ============================================================
# [최초 1회 실행] 필요 패키지 설치
# ============================================================
!pip install xgboost lightgbm scikit-learn matplotlib seaborn joblib

In [ ]:
# ============================================================
# test.py 연동 (test.py 수정 시 자동 반영)
# ============================================================
import importlib
import test

# test.py가 수정되었으면 다시 로드
importlib.reload(test)

from test import CONFIG, main, load_data, RENAME_MAP
from test import prepare_monthly_target, create_features
from test import get_models, split_timeseries, train_all_models
from test import recursive_forecast, save_best_model

print("✅ test.py 연동 완료!\n")
print("=" * 50)
print("현재 CONFIG 설정:")
print("=" * 50)
for k, v in CONFIG.items():
    if k not in ("model_params", "data_files", "ensemble"):
        print(f"  {k}: {v}")

print(f"\n활성 모델:")
for name, on in CONFIG["models"].items():
    if on:
        print(f"  ✓ {name}")

print(f"\n평가지표: MAE, MAPE, MSE, RMSE (MAE 기준 최적 모델 선정)")
print(f"모델 저장: {CONFIG.get('model_dir', './model')}")
print(f"학습 이력: {CONFIG.get('history_dir', './model/history')}")

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 198.0 kB/s eta 0:08:31
   ---------------------------------------- 0.5/101.7 MB 198.0 kB/s eta 0:08:31
   ---------------------------------------- 0.5/101.7 MB 19

# 앙상블 융합 모델 — 자금예측
- **모델**: XGBoost + LightGBM + GradientBoosting (개별) + Voting + Stacking (융합)
- **타겟**: df7(월별_매출정보)의 `TOT_MN_MNAM`(총공급가액) 월별 합계
- **예측기간**: 2025.02 ~ 2025.07 (6개월)
- **평가지표**: MAE ★(기준), MAPE, MSE, RMSE
- **모델 관리**: 최적 모델 → `model/`, 학습 이력 → `model/history/`
- **연동**: `test.py` 수정 시 Cell 2 재실행으로 자동 반영

In [ ]:
# ============================================================
# [데이터 확인] 회사 수 + 기간 확인
# - 데이터 원본 행(row)은 출력하지 않습니다
# ============================================================
data = load_data()

print("\n📋 데이터 요약:")
print("=" * 70)
for name, df in data.items():
    biz_count = df['NO_BIZ'].nunique() if 'NO_BIZ' in df.columns else 'N/A'
    rows = len(df)
    cols = len(df.columns)
    
    period = ''
    if 'DM_DATA' in df.columns:
        period = f"{df['DM_DATA'].min()} ~ {df['DM_DATA'].max()}"
    elif 'DW_BAS_NYYMM' in df.columns:
        vals = sorted(df['DW_BAS_NYYMM'].unique())
        period = f"기준월: {vals[0]}~{vals[-1]}" if len(vals) > 1 else f"기준월: {vals[0]}"
    
    print(f"  {name:12s} | {rows:>7,}행 × {cols}열 | 사업자수={biz_count} | {period}")

# 전체 고유 사업자 수
all_biz = set()
for df in data.values():
    if 'NO_BIZ' in df.columns:
        all_biz.update(df['NO_BIZ'].unique())
print(f"\n🏢 전체 고유 사업자(회사) 수: {len(all_biz)}개")

In [ ]:
# ============================================================
# [실행] 전체 ML 파이프라인 (test.py main 함수 호출)
# ============================================================
# test.py의 CONFIG 설정대로 실행됩니다.
# CONFIG를 변경하려면 test.py를 수정한 뒤 Cell 2를 다시 실행하세요.
#
# 실행 순서:
#   1. 데이터 로드 + 컬럼 변환
#   2. 월별 시계열 집계
#   3. 피처 엔지니어링 (lag, rolling, pct_change)
#   4. 학습/검증 분할
#   5. 7개 모델 학습 & 평가 (XGB, LGB, GBM, RF, Ridge, Voting, Stacking)
#   6. 최적 모델 저장 (model/) + 학습 이력 (model/history/)
#   7. 전체 데이터 재학습 → 미래 6개월 재귀 예측
#   8. 시각화 + 결과 저장 (result/ 폴더)
#
# 평가지표: MAE ★(기준), MAPE, MSE, RMSE

results_df, pred_dfs, trained_models = main()

In [ ]:
# ============================================================
# [결과] 모델별 예측 비교표
# ============================================================
import pandas as pd

target = CONFIG["target_col"]
comparison = None
for name, pdf in pred_dfs.items():
    cols = [CONFIG["date_col"], f'predicted_{target}']
    tmp = pdf[cols].rename(columns={f'predicted_{target}': name})
    if comparison is None:
        comparison = tmp
    else:
        comparison = comparison.merge(tmp, on=CONFIG["date_col"])

if comparison is not None:
    # 모델 평균 열 추가
    model_cols = [c for c in comparison.columns if c != CONFIG["date_col"]]
    comparison['모델평균'] = comparison[model_cols].mean(axis=1)
    
    print("📊 모델별 6개월 예측 비교 (원):")
    print("=" * 120)
    
    # 숫자 포맷팅
    fmt = comparison.copy()
    for col in model_cols + ['모델평균']:
        fmt[col] = fmt[col].apply(lambda x: f'{x:>18,.0f}')
    print(fmt.to_string(index=False))
    
    print("\n" + "=" * 120)
    # 6개월 합계
    print("\n📌 6개월 합계:")
    for col in model_cols + ['모델평균']:
        total = comparison[col].sum() if col != '모델평균' else comparison['모델평균'].sum()
        print(f"  {col:25s}: {total:>20,.0f} 원")